# **MINI BOOTCAMP DATA ENGINEERING PROJECT PART 2**

---


*DIGITAL SKILL FAIR 44.0: FACULTY OF DATA ENGINEERING*

*   **Project Name: End-to-end Job Market Trends Pipeline**
*   **by Fauzan Dharmawan**

---



**⚠️ Disclaimer**

This project is created **solely for educational and learning purposes** .

All data collected and analyzed in this project are **limited to publicly accessible information** and comply with each website’s robots.txt policy.

The dataset used in this project only covers the **period of up to 30 days ago, and no personal, confidential, or sensitive user data** are collected, stored, or shared.

This project does **not represent**, nor is it **affiliated** with LinkedIn in any form.

Check robot.txt:
https://www.linkedin.com/robots.txt



---



**In the transform stage**, Apache Spark is used to process the staging data stored in Microsoft SQL Server. Spark connects to SQL Server using JDBC and performs data cleaning and transformation operations.

The transformation process includes:

*   Removing duplicate job records based on job_id
*   Handling missing values using fillna() and dropna()
*   Standardizing text fields (lowercase, trimming spaces)
*   Casting columns into appropriate data types


*Spark was chosen because it is a scalable distributed data processing framework commonly used in modern data engineering pipelines.*

In [308]:
!pip install pyspark


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## **Connect Database**

In [309]:
pip install sqlalchemy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [310]:
import pandas as pd
from sqlalchemy import create_engine

# Create engine
engine = create_engine(
    "mssql+pyodbc://DESKTOP-APH7COF\\SQLEXPRESS/recent_job_market"
    "?driver=ODBC+Driver+17+for+SQL+Server"
    "&trusted_connection=yes"
)

# Read data
query = "SELECT * FROM dbo.recent_jobs"

pdf = pd.read_sql(query, engine)

print(pdf.head())

D:\python 3.10\lib\site-packages\pandas\io\sql.py:1648: SAWarning: Unrecognized server version info '17.0.1000.7'.  Some SQL Server features may not function properly.
  con = self.exit_stack.enter_context(con.connect())


  linkedinJob_id                                 linkedinJob_title  \
0     4375724428                          Finance Accounting Staff   
1     4370258070  Officer Development Program (ODP) Retail Banking   
2     4376705498                             Staff GA (Pulogadung)   
3     4375151664                                      Import Staff   
4     4368981113                Future Bankers Development Program   

                        linkedinCompany_name     linkedinJob_location  \
0  Kalbe Nutritionals (PT Sanghiang Perkasa)           Karawang Barat   
1   PT. BANK NEGARA INDONESIA (Persero) Tbk.  Jakarta Raya, Indonesia   
2                PT. Sarimelati Kencana Tbk.            Jakarta Timur   
3               PT. Surya TOTO Indonesia Tbk         Area DKI Jakarta   
4                       PT Bank Sinarmas Tbk  Jakarta Raya, Indonesia   

  linkedinJob_seniority linkedinJob_employmentType  \
0        Tingkat pemula                Penuh waktu   
1        Tingkat pemula         

## **Spark Config**

In [311]:
import os
from pyspark.sql import SparkSession

# Replace this with YOUR python.exe path
python_path = r"D:\python 3.10\python.exe"

os.environ["PYSPARK_PYTHON"] = python_path
os.environ["PYSPARK_DRIVER_PYTHON"] = python_path

spark = SparkSession.builder \
    .appName("ETLPipeline") \
    .master("local[*]") \
    .getOrCreate()

spark_df = spark.createDataFrame(pdf)


In [312]:
spark_df.printSchema()

root
 |-- linkedinJob_id: string (nullable = true)
 |-- linkedinJob_title: string (nullable = true)
 |-- linkedinCompany_name: string (nullable = true)
 |-- linkedinJob_location: string (nullable = true)
 |-- linkedinJob_seniority: string (nullable = true)
 |-- linkedinJob_employmentType: string (nullable = true)
 |-- linkedinJob_function: string (nullable = true)
 |-- linkedinJob_industries: string (nullable = true)
 |-- linkedinNumber_applicants: string (nullable = true)
 |-- linkedinJob_postedTime: string (nullable = true)



In [313]:
spark_df.show(5)

+--------------+--------------------+--------------------+--------------------+---------------------+--------------------------+--------------------+----------------------+-------------------------+----------------------+
|linkedinJob_id|   linkedinJob_title|linkedinCompany_name|linkedinJob_location|linkedinJob_seniority|linkedinJob_employmentType|linkedinJob_function|linkedinJob_industries|linkedinNumber_applicants|linkedinJob_postedTime|
+--------------+--------------------+--------------------+--------------------+---------------------+--------------------------+--------------------+----------------------+-------------------------+----------------------+
|    4375724428|Finance Accountin...|Kalbe Nutritional...|      Karawang Barat|       Tingkat pemula|               Penuh waktu|Akuntansi/Audit, ...|  Jasa Penyedia Mak...|                     NULL|      4 hari yang lalu|
|    4370258070|Officer Developme...|PT. BANK NEGARA I...|Jakarta Raya, Ind...|       Tingkat pemula|           

## **Clean the Data**

In [314]:
from pyspark.sql.functions import col

spark_df = spark_df.filter(col("linkedinJob_id").isNotNull())
spark_df = spark_df.dropDuplicates(["linkedinJob_id"])

## **Normalize Timestamp**

In [315]:
from pyspark.sql import functions as F

spark.conf.set("spark.sql.session.timeZone", "Asia/Jakarta")

spark_df = spark_df \
    .withColumn("amount",
        F.regexp_extract("linkedinJob_postedTime", r"(\d+)", 1).cast("int")
    ) \
    .withColumn("unit",
        F.regexp_extract("linkedinJob_postedTime", r"\d+\s+(\w+)", 1)
    ) \
    .withColumn("seconds_to_subtract",
        F.when(F.col("unit") == "menit", F.col("amount") * 60)
         .when(F.col("unit") == "jam", F.col("amount") * 3600)
         .when(F.col("unit") == "hari", F.col("amount") * 86400)
         .when(F.col("unit") == "minggu", F.col("amount") * 604800)
         .when(F.col("unit") == "bulan", F.col("amount") * 2592000)
         .when(F.col("unit") == "tahun", F.col("amount") * 31536000)
    ) \
    .withColumn(
        "linkedinJob_postedTime",
        (
            F.current_timestamp().cast("long") - F.col("seconds_to_subtract")
        ).cast("timestamp")
    ) \
    .drop("amount", "unit", "seconds_to_subtract")

## **Change Data Type**

In [316]:
#For linkedinNumber_applicants
from pyspark.sql.functions import col, regexp_extract

spark_df = spark_df.withColumn(
    "linkedinNumber_applicants",
    regexp_extract(col("linkedinNumber_applicants"), r"(\d+)", 1).cast("int")
)

In [317]:
spark_df.printSchema()

root
 |-- linkedinJob_id: string (nullable = true)
 |-- linkedinJob_title: string (nullable = true)
 |-- linkedinCompany_name: string (nullable = true)
 |-- linkedinJob_location: string (nullable = true)
 |-- linkedinJob_seniority: string (nullable = true)
 |-- linkedinJob_employmentType: string (nullable = true)
 |-- linkedinJob_function: string (nullable = true)
 |-- linkedinJob_industries: string (nullable = true)
 |-- linkedinNumber_applicants: integer (nullable = true)
 |-- linkedinJob_postedTime: timestamp (nullable = true)



## **Normalize Job Location**

In [318]:
#Check Data
spark_df.select("linkedinJob_location").show(100, truncate=False)

+----------------------------+
|linkedinJob_location        |
+----------------------------+
|Jakarta Raya, Indonesia     |
|Jakarta Raya, Indonesia     |
|Jakarta                     |
|Jakarta                     |
|Kota Bekasi                 |
|Jakarta Raya, Indonesia     |
|Jakarta Raya, Indonesia     |
|Jakarta Raya, Indonesia     |
|Jakarta Raya, Indonesia     |
|Pontianak                   |
|Jakarta Raya, Indonesia     |
|Jakarta Raya, Indonesia     |
|Jakarta Raya, Indonesia     |
|Jakarta Selatan             |
|Sulawesi Barat, Indonesia   |
|Kota Bekasi                 |
|Batam dan Sekitarnya        |
|Jakarta Raya, Indonesia     |
|Jakarta Raya, Indonesia     |
|Jakarta Raya, Indonesia     |
|Jakarta Raya, Indonesia     |
|Jakarta Raya, Indonesia     |
|Jakarta Raya, Indonesia     |
|Jakarta Raya, Indonesia     |
|Jakarta Raya, Indonesia     |
|Area DKI Jakarta            |
|Jakarta Raya, Indonesia     |
|Jakarta Raya, Indonesia     |
|Jakarta Raya, Indonesia     |
|Jakarta

In [319]:
#Clean the Location Column
from pyspark.sql.functions import col, lower, trim, regexp_replace

spark_df = spark_df.withColumn(
    "location_clean",
    trim(lower(col("linkedinJob_location")))
)

patterns = [
    ", indonesia",
    " indonesia",
    "area ",
    " dan sekitarnya",
    " raya",
    "kota "
]

for p in patterns:
    spark_df = spark_df.withColumn(
        "location_clean",
        regexp_replace(col("location_clean"), p, "")
    )

In [320]:
#City
from pyspark.sql.functions import when, initcap

spark_df = spark_df.withColumn(
    "linkedinJob_city",
    when(col("location_clean").contains("jakarta"), "Jakarta")
    .when(col("location_clean").contains("gambir"), "Jakarta")
    .when(col("location_clean").contains("bandung"), "Bandung")
    .when(col("location_clean").contains("surabaya"), "Surabaya")
    .when(col("location_clean").contains("bekasi"), "Bekasi")
    .when(col("location_clean").contains("tangerang selatan"), "Tangerang Selatan")
    .when(col("location_clean").contains("tangerang"), "Tangerang")
    .when(col("location_clean").contains("karawang"), "Karawang")
    .otherwise(initcap(col("location_clean")))
)

In [321]:
#Mapping Logic Provience
spark_df = spark_df.withColumn(
    "linkedinJob_province",
    when(col("linkedinJob_city") == "Jakarta", "DKI Jakarta")
    .when(col("linkedinJob_city").isin("Bandung", "Bekasi", "Karawang"), "Jawa Barat")
    .when(col("linkedinJob_city").isin("Tangerang", "Tangerang Selatan"), "Banten")
    .when(col("linkedinJob_city") == "Surabaya", "Jawa Timur")
    .otherwise(None)
)

In [322]:
#Country
from pyspark.sql.functions import lit

spark_df = spark_df.withColumn(
    "linkedinJob_country",
    lit("Indonesia")
)

In [323]:
#Drop Column
spark_df = spark_df.drop("linkedinJob_location", "location_clean")

In [324]:
spark_df.printSchema()

root
 |-- linkedinJob_id: string (nullable = true)
 |-- linkedinJob_title: string (nullable = true)
 |-- linkedinCompany_name: string (nullable = true)
 |-- linkedinJob_seniority: string (nullable = true)
 |-- linkedinJob_employmentType: string (nullable = true)
 |-- linkedinJob_function: string (nullable = true)
 |-- linkedinJob_industries: string (nullable = true)
 |-- linkedinNumber_applicants: integer (nullable = true)
 |-- linkedinJob_postedTime: timestamp (nullable = true)
 |-- linkedinJob_city: string (nullable = true)
 |-- linkedinJob_province: string (nullable = true)
 |-- linkedinJob_country: string (nullable = false)



In [325]:
spark_df.show(5)

+--------------+--------------------+--------------------+---------------------+--------------------------+--------------------+----------------------+-------------------------+----------------------+----------------+--------------------+-------------------+
|linkedinJob_id|   linkedinJob_title|linkedinCompany_name|linkedinJob_seniority|linkedinJob_employmentType|linkedinJob_function|linkedinJob_industries|linkedinNumber_applicants|linkedinJob_postedTime|linkedinJob_city|linkedinJob_province|linkedinJob_country|
+--------------+--------------------+--------------------+---------------------+--------------------------+--------------------+----------------------+-------------------------+----------------------+----------------+--------------------+-------------------+
|    4046132512|Competitor Intell...|              Shopee| Senior tingkat me...|               Penuh waktu|Strategi/Perencan...|         Jasa Keuangan|                     NULL|   2026-02-23 08:44:11|         Jakarta|      

*After transformation, the cleaned dataset is written to PostgreSQL as the warehouse layer.*

## **Load to Bigquery**

In [326]:
#If you want to download your data
pandas_df = spark_df.toPandas()
pandas_df.to_excel("linkedin_jobs_cleaned.xlsx", index=False)

In [327]:
!pip install google-cloud-bigquery


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [328]:
pandas_df = spark_df.toPandas()

pandas_df.to_csv("linkedin_jobs_cleaned.csv", index=False)

In [329]:
from google.cloud import bigquery
from google.oauth2 import service_account

credentials = service_account.Credentials.from_service_account_file(
    "mini-bootcamp-data-engineering-f6b829fb22d4.json"
)

client = bigquery.Client(credentials=credentials)

table_id = "mini-bootcamp-data-engineering.recent_job_market.recent_jobs"

job_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.CSV,
    skip_leading_rows=1,
    autodetect=True,
)

with open("linkedin_jobs_cleaned.csv", "rb") as source_file:
    job = client.load_table_from_file(
        source_file,
        table_id,
        job_config=job_config
    )

job.result()

print(" CSV successfully loaded to BigQuery.")

 CSV successfully loaded to BigQuery.
